# MPLADS — Data Audit

This notebook performs the first reproducible audit of the official Lok Sabha and Rajya Sabha MPLADS CSV files.

**Goal:** understand data quality before feature engineering or ML.

We will:
1. Load all source CSVs.
2. Inspect shapes and columns.
3. Standardize Work IDs.
4. Check duplicate/missing IDs.
5. Check missing values.
6. Check dates and chronological consistency.
7. Check monetary values and financial consistency.
8. Inspect status/category distributions.
9. Record findings without deleting legitimate anomalies.

> **Important:** unusual values are not automatically removed. This project is specifically intended to detect unusual patterns later.


In [12]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..")

RAW = PROJECT_ROOT / "data" / "raw"

print("Raw data directory:", RAW.resolve())
print("Exists:", RAW.exists())


Raw data directory: D:\SIH\mplads-ai-risk-platform\data\raw
Exists: True


In [13]:
# Expected files
FILES = {
    "ls_allocated": RAW / "lok_sabha" / "Allocated Limit for Honble MPs Lok Sabha.csv",
    "ls_recommended": RAW / "lok_sabha" / "Works Recommended Lok Sabha.csv",
    "ls_sanctioned": RAW / "lok_sabha" / "Works Sanctioned Lok Sabha.csv",
    "ls_completed": RAW / "lok_sabha" / "Works Completed Lok Sabha.csv",
    "ls_expenditure": RAW / "lok_sabha" / "Expenditure on Completed and On-going Works as on Date Lok Sabha.csv",

    "rs_allocated": RAW / "rajya_sabha" / "Allocated Limit for Honble MPs Rajya Sabha.csv",
    "rs_recommended": RAW / "rajya_sabha" / "Works Recommended Rajya Sabha.csv",
    "rs_sanctioned": RAW / "rajya_sabha" / "Works Sanctioned Rajya Sabha.csv",
    "rs_completed": RAW / "rajya_sabha" / "Works Completed Rajya Sabha.csv",
    "rs_expenditure": RAW / "rajya_sabha" / "Expenditure on Completed and On-going Works as on Date Rajya Sabha.csv",
}

for name, path in FILES.items():
    print(f"{name:16} | exists={path.exists()} | {path}")


ls_allocated     | exists=True | ..\data\raw\lok_sabha\Allocated Limit for Honble MPs Lok Sabha.csv
ls_recommended   | exists=True | ..\data\raw\lok_sabha\Works Recommended Lok Sabha.csv
ls_sanctioned    | exists=True | ..\data\raw\lok_sabha\Works Sanctioned Lok Sabha.csv
ls_completed     | exists=True | ..\data\raw\lok_sabha\Works Completed Lok Sabha.csv
ls_expenditure   | exists=True | ..\data\raw\lok_sabha\Expenditure on Completed and On-going Works as on Date Lok Sabha.csv
rs_allocated     | exists=True | ..\data\raw\rajya_sabha\Allocated Limit for Honble MPs Rajya Sabha.csv
rs_recommended   | exists=True | ..\data\raw\rajya_sabha\Works Recommended Rajya Sabha.csv
rs_sanctioned    | exists=True | ..\data\raw\rajya_sabha\Works Sanctioned Rajya Sabha.csv
rs_completed     | exists=True | ..\data\raw\rajya_sabha\Works Completed Rajya Sabha.csv
rs_expenditure   | exists=True | ..\data\raw\rajya_sabha\Expenditure on Completed and On-going Works as on Date Rajya Sabha.csv


In [14]:
def read_csv(path):
    return pd.read_csv(path, low_memory=False)

dfs = {name: read_csv(path) for name, path in FILES.items()}

summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_full_rows": int(df.duplicated().sum())
    }
    for name, df in dfs.items()
])

summary


,dataset,rows,columns,duplicate_full_rows
0,ls_allocated,544,5,0
1,ls_recommended,106440,11,0
2,ls_sanctioned,79083,12,0
3,ls_completed,34276,11,0
4,ls_expenditure,83981,11,0
5,rs_allocated,232,5,0
6,rs_recommended,25186,11,0
7,rs_sanctioned,19565,12,0
8,rs_completed,9957,11,0
9,rs_expenditure,25119,11,0


In [15]:
def normalize_work_id(value):
    if pd.isna(value):
        return None
    text = str(value).replace("\t", " ").strip()
    match = re.search(r"(WS/\s*MP\d+/\d{4}-\d{4}/\d+)", text, re.IGNORECASE)
    if not match:
        return None
    return re.sub(r"\s+", "", match.group(1)).upper()

def money_to_numeric(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("₹", "", regex=False)
        .str.strip(),
        errors="coerce"
    )


## Work ID audit

In [16]:
ID_COLUMNS = {
    "recommended": "WORK",
    "sanctioned": "Work",
    "completed": "Work",
    "expenditure": "Work ID",
}

for house in ["Lok Sabha", "Rajya Sabha"]:
    print(f"\n### {house}")
    for kind, column in ID_COLUMNS.items():
        key = f"{'ls' if house == 'Lok Sabha' else 'rs'}_{kind}"
        df = dfs[key].copy()
        ids = df[column].map(normalize_work_id)

        print(
            f"{kind:12} rows={len(df):7,} | "
            f"unique_ids={ids.nunique():7,} | "
            f"missing_ids={ids.isna().sum():6,} | "
            f"duplicate_id_rows={ids.duplicated(keep=False).sum():6,}"
        )



### Lok Sabha
recommended  rows=106,440 | unique_ids= 78,720 | missing_ids=27,720 | duplicate_id_rows=27,720
sanctioned   rows= 79,083 | unique_ids= 79,082 | missing_ids=     1 | duplicate_id_rows=     0
completed    rows= 34,276 | unique_ids= 34,275 | missing_ids=     1 | duplicate_id_rows=     0
expenditure  rows= 83,981 | unique_ids= 56,502 | missing_ids=     1 | duplicate_id_rows=42,070

### Rajya Sabha
recommended  rows= 25,186 | unique_ids= 19,338 | missing_ids= 5,848 | duplicate_id_rows= 5,848
sanctioned   rows= 19,565 | unique_ids= 19,564 | missing_ids=     1 | duplicate_id_rows=     0
completed    rows=  9,957 | unique_ids=  9,956 | missing_ids=     1 | duplicate_id_rows=     0
expenditure  rows= 25,119 | unique_ids= 15,307 | missing_ids=     1 | duplicate_id_rows=14,752


## Missing-value audit

In [17]:
for name, df in dfs.items():
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print(f"\n### {name}")
    if len(missing) == 0:
        print("No missing values.")
    else:
        print(missing)



### ls_allocated
Allocated AMOUNT ( ₹ )    1
dtype: int64

### ls_recommended
Sanction Date       27719
Work description      115
dtype: int64

### ls_sanctioned
Work description    98
dtype: int64

### ls_completed
Image                     9303
Amount Disbursed ( ₹ )      84
Work Description            79
dtype: int64

### ls_expenditure
No missing values.

### rs_allocated
No missing values.

### rs_recommended
Sanction Date       5847
Work description      23
Work category          5
dtype: int64

### rs_sanctioned
Work description    11
Work category        7
dtype: int64

### rs_completed
Image                     3458
Amount Disbursed ( ₹ )      18
Work Description             6
Work Category                5
dtype: int64

### rs_expenditure
No missing values.


## Sanctioned-work data quality

In [18]:
VALID_STATUSES = {
    "Physical Inspection",
    "Sanction",
    "Vendor Identification",
    "Work partially Completed",
    "Work Completed",
    "Time Estimation",
}

for house in ["Lok Sabha", "Rajya Sabha"]:
    key = "ls_sanctioned" if house == "Lok Sabha" else "rs_sanctioned"
    df = dfs[key].copy()

    # Remove summary/footer rows from analytical checks.
    analytical = df[df["Work Status"].isin(VALID_STATUSES)].copy()

    analytical["recommended_date"] = pd.to_datetime(
        analytical["Recommended date"], errors="coerce", dayfirst=True
    )
    analytical["sanction_date"] = pd.to_datetime(
        analytical["Sanction Date"], errors="coerce", dayfirst=True
    )
    analytical["sanction_amount"] = money_to_numeric(
        analytical["Sanction Amount ( ₹ )"]
    )

    print(f"\n### {house}")
    print("Analytical rows:", len(analytical))
    print("Invalid/summary status rows excluded:", len(df) - len(analytical))
    print("Recommendation after sanction:", (
        (analytical["recommended_date"] > analytical["sanction_date"])
        & analytical["recommended_date"].notna()
        & analytical["sanction_date"].notna()
    ).sum())
    print("Missing sanction amount:", analytical["sanction_amount"].isna().sum())
    print("Zero/negative sanction amount:", (analytical["sanction_amount"] <= 0).sum())
    print("Status distribution:")
    print(analytical["Work Status"].value_counts())



### Lok Sabha
Analytical rows: 79082
Invalid/summary status rows excluded: 1
Recommendation after sanction: 0
Missing sanction amount: 0
Zero/negative sanction amount: 0
Status distribution:
Work Status
Physical Inspection         34216
Sanction                    20222
Vendor Identification       11608
Work partially Completed     8030
Work Completed               4184
Time Estimation               822
Name: count, dtype: int64

### Rajya Sabha
Analytical rows: 19564
Invalid/summary status rows excluded: 1
Recommendation after sanction: 0
Missing sanction amount: 0
Zero/negative sanction amount: 0
Status distribution:
Work Status
Physical Inspection         9860
Sanction                    3925
Vendor Identification       2814
Work partially Completed    1887
Work Completed               974
Time Estimation              104
Name: count, dtype: int64


## Financial and chronology consistency

In [19]:
def prepare_project_checks(house):
    prefix = "ls" if house == "Lok Sabha" else "rs"

    san = dfs[f"{prefix}_sanctioned"].copy()
    san = san[san["Work Status"].isin(VALID_STATUSES)].copy()
    san["project_id"] = san["Work"].map(normalize_work_id)
    san["sanction_date"] = pd.to_datetime(san["Sanction Date"], errors="coerce", dayfirst=True)
    san["sanction_amount"] = money_to_numeric(san["Sanction Amount ( ₹ )"])

    comp = dfs[f"{prefix}_completed"].copy()
    comp["project_id"] = comp["Work"].map(normalize_work_id)
    comp["completion_date"] = pd.to_datetime(comp["Completion Date"], errors="coerce", dayfirst=True)
    comp["completed_amount"] = money_to_numeric(comp["Amount Disbursed ( ₹ )"])
    comp = comp.dropna(subset=["project_id"])
    comp = comp.groupby("project_id").agg(
        completion_date=("completion_date", "max"),
        completed_amount=("completed_amount", "sum"),
    )

    exp = dfs[f"{prefix}_expenditure"].copy()
    exp["project_id"] = exp["Work ID"].map(normalize_work_id)
    exp["expenditure_date"] = pd.to_datetime(exp["Expenditure Date"], errors="coerce", dayfirst=True)
    exp["expenditure_amount"] = money_to_numeric(exp["Fund Disbursed Amount ( ₹ )"])
    exp = exp.dropna(subset=["project_id"])
    exp = exp.groupby("project_id").agg(
        total_expenditure=("expenditure_amount", "sum"),
        first_expenditure_date=("expenditure_date", "min"),
        last_expenditure_date=("expenditure_date", "max"),
        payment_count=("expenditure_amount", "size"),
        vendor_count=("Vendor Name", "nunique"),
    )

    project = san[["project_id", "sanction_date", "sanction_amount"]].drop_duplicates("project_id")
    project = project.merge(comp, on="project_id", how="left")
    project = project.merge(exp, on="project_id", how="left")
    return project

for house in ["Lok Sabha", "Rajya Sabha"]:
    p = prepare_project_checks(house)

    completion_before_sanction = (
        (p["completion_date"] < p["sanction_date"])
        & p["completion_date"].notna()
        & p["sanction_date"].notna()
    ).sum()

    expenditure_above_sanction = (
        (p["total_expenditure"] > p["sanction_amount"])
        & p["total_expenditure"].notna()
        & p["sanction_amount"].notna()
    ).sum()

    completed_amount_above_sanction = (
        (p["completed_amount"] > p["sanction_amount"])
        & p["completed_amount"].notna()
        & p["sanction_amount"].notna()
    ).sum()

    print(f"\n### {house}")
    print("Projects:", len(p))
    print("Completion before sanction:", completion_before_sanction)
    print("Total expenditure > sanction amount:", expenditure_above_sanction)
    print("Completed amount > sanction amount:", completed_amount_above_sanction)
    print("Projects with expenditure:", p["total_expenditure"].notna().sum())
    print("Projects with completion:", p["completion_date"].notna().sum())



### Lok Sabha
Projects: 79082
Completion before sanction: 0
Total expenditure > sanction amount: 0
Completed amount > sanction amount: 0
Projects with expenditure: 56502
Projects with completion: 34275

### Rajya Sabha
Projects: 19564
Completion before sanction: 0
Total expenditure > sanction amount: 1
Completed amount > sanction amount: 0
Projects with expenditure: 15306
Projects with completion: 9956


## Audit conclusions to record

Based on the current source files:

- The sanctioned-work table is the most suitable backbone for a project-level canonical dataset because it contains a clean Work identifier for essentially every analytical row.
- Expenditure is a **one-to-many** relationship: one project can have multiple payment/expenditure rows. It must therefore be aggregated before joining to project-level data.
- Missing completion does **not** mean delayed; it may simply mean the project is ongoing or has no completion record.
- Missing expenditure does **not** automatically indicate a financial anomaly.
- The sanctioned files contain a final **Grand Total** row that must be excluded from project-level analysis.
- Some recommended-work rows do not contain a usable Work ID and therefore should not be assigned artificial IDs.
- Legitimate extreme monetary values must be preserved for later anomaly detection.
